# US-012 - EDA bivariado, multivariado y temporal sobre PASTIS-R

**Avance**: A1 — EDA (Tec MNA · Proyecto Integrador)
**EPIC**: E2 (Data Understanding)
**Rubrica**: preguntas 6, 7 y 10 de la rubrica EDA del Avance 1.
**Autores**: Arthur Zizumbo, Aaron Bocanegra, Isaac Avila.

Notebook secuencial en 10 secciones (papermill-ready):

1. Setup + carga PASTIS-R sample estratificado.
2. Computo subset 6 indices espectrales (NDVI, NDWI, NDMI, EVI, SAVI, NDRE).
3. Correlacion bandas x indices (Pearson + Spearman) - AC-2.
4. VIF multicolinealidad - AC-3.
5. Pairplot top-5 features por clase - AC-4.
6. Pico NDVI por clase + timing - AC-5.
7. ACF/PACF por parcela y clase - AC-6.
8. DTW clustering temporal mono vs doble ciclo - AC-7.
9. ERA5 anomalias 2018-2020 cruzado con NDVI maximo - AC-8 (opcional).
10. Conclusiones para EPIC 3 - AC-10.

## Requisitos para ejecucion end-to-end

- **PASTIS-R completo descomprimido** en `data/PASTIS-R/` (DATA_S2, ANNOTATIONS,
  metadata.geojson con campo `dates-S2`).
- **Opcional para AC-8**: `earthengine authenticate` ejecutado para consumir
  ERA5-Land via GEE.

**Sin estos requisitos el notebook degrada graciosamente**: las funciones de
`ml/ingest/pastis_loader.py` y `ml/ingest/gee_sampler.py` devuelven DataFrames
vacios con esquema valido; las secciones afectadas registran un mensaje claro y
el notebook completa `exit 0` bajo `papermill`. Los plots correspondientes
muestran placeholders 'Sin datos'.

In [ ]:
# Parametros papermill (sobreescribibles con -p)
n_parcels = 200
n_pastis_patches = 30
max_lag = 6
n_dtw_clusters = 4
era5_years = [2018, 2019, 2020]
figures_dir = "paper/figures/us-012"
seed = 42

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from ml.analysis.correlations import (
    SPECTRAL_INDICES_CORE,
    acf_pacf_per_parcel,
    compute_indices_subset,
    correlation_pair,
    dtw_cluster_temporal,
    era5_ndvi_anomaly,
    phenology_peaks,
    vif_table,
)
from ml.analysis.visualization import (
    acf_grid_by_class,
    correlation_heatmap,
    dtw_centroids_plot,
    dual_axis_precip_ndvi,
    pairplot_by_class,
    vif_barplot,
)
from ml.ingest.gee_sampler import era5_annual_precip
from ml.ingest.pastis_loader import (
    PASTIS_R_CLASSES,
    PASTIS_S2_BANDS,
    pastis_patch_coords,
    pastis_to_polars,
)

REPO = Path.cwd() if Path.cwd().name == "agro_sat_copilot" else Path.cwd().parent
FIGURES = REPO / figures_dir
FIGURES.mkdir(parents=True, exist_ok=True)
CACHE = REPO / "data" / "cache" / "gee"
CACHE.mkdir(parents=True, exist_ok=True)
PASTIS_ROOT = REPO / "data" / "PASTIS-R"
print(f"Figures dir: {FIGURES}")
print(f"PASTIS root: {PASTIS_ROOT} (exists={PASTIS_ROOT.exists()})")
print(f"Subset indices: {list(SPECTRAL_INDICES_CORE.keys())}")

## Seccion 1 - Carga PASTIS-R sample estratificado

Descarga PASTIS-R no esta scriptada aqui (R-8 resuelto en planning): se asume
que `data/PASTIS-R/` ya esta descomprimido. Si no, las celdas downstream
muestran `df vacio` y completan limpio.

In [ ]:
pastis_meta = PASTIS_ROOT / "metadata.geojson"
patch_coords = pastis_patch_coords(pastis_meta)
print(f"PASTIS patches: {patch_coords.height}")
if patch_coords.is_empty():
    print("PASTIS-R no disponible - secciones temporales degradan graceful.")
patch_ids_sample = patch_coords.head(n_pastis_patches)["patch_id"].to_list() if not patch_coords.is_empty() else []
print(f"Patches a cargar (cap n_pastis_patches): {len(patch_ids_sample)}")

In [ ]:
# Cargar pixeles con stride 8 para limitar memoria (128*128 / 64 = 256 por patch x banda x T)
df_long = pastis_to_polars(
    patch_ids=patch_ids_sample,
    bands=PASTIS_S2_BANDS,
    root=PASTIS_ROOT,
    include_labels=True,
    include_dates=True,
    pixel_stride=8,
) if patch_ids_sample else pl.DataFrame()
print(f"Long-format rows: {df_long.height}")
if not df_long.is_empty():
    print(df_long.head(5))

In [ ]:
# Pivot long -> wide para tener una columna por banda y subsample por parcela
# Definimos `parcel_id` como (patch_id, y, x) que en PASTIS-R representa un pixel-parcela
if not df_long.is_empty():
    df_wide = (
        df_long.with_columns(
            (pl.col("patch_id").cast(pl.Utf8) + "_" + pl.col("y").cast(pl.Utf8) + "_" + pl.col("x").cast(pl.Utf8)).alias("parcel_id")
        )
        .pivot(values="value", index=["parcel_id", "patch_id", "t", "date", "y", "x", "class_id", "class_name", "fold"], on="band")
        .with_columns(pl.col("date").cast(pl.Utf8).str.to_date("%Y%m%d", strict=False).alias("date_parsed"))
    )
    # Estratificar muestreo a n_parcels parcelas-pixel unicas con clases [1, 18]
    parcels_all = (
        df_wide.filter(pl.col("class_id").is_between(1, 18))
        .select(["parcel_id", "class_id", "class_name"]).unique(subset=["parcel_id"]).sort("parcel_id")
    )
    if parcels_all.height > n_parcels:
        parcels_sel = parcels_all.sample(n=n_parcels, seed=seed, shuffle=True)
    else:
        parcels_sel = parcels_all
    df_pivot = df_wide.filter(pl.col("parcel_id").is_in(parcels_sel["parcel_id"].to_list()))
    print(f"Parcelas-pixel seleccionadas: {parcels_sel.height}")
    print(f"Filas wide: {df_pivot.height}")
    print("Top clases en muestra:")
    print(parcels_sel.group_by("class_name").agg(pl.len().alias("n")).sort("n", descending=True).head(10))
else:
    df_pivot = pl.DataFrame()
    print("Sin PASTIS-R: skipeando secciones 2-9")

## Seccion 2 - Subset 6 indices espectrales

**Disclaimer**: este notebook implementa el subset core de 6 indices. La
biblioteca completa de 17 indices se entrega en US-014 (Avance 2).

In [ ]:
if not df_pivot.is_empty():
    df_idx = compute_indices_subset(df_pivot)
    indices_present = [c for c in SPECTRAL_INDICES_CORE if c in df_idx.columns]
    print(f"Indices calculados: {indices_present}")
    desc = df_idx.select(indices_present).describe()
    print(desc)
else:
    df_idx = pl.DataFrame()
    indices_present = []

## Seccion 3 - Correlacion bandas x indices (AC-2)

Heatmaps Pearson + Spearman entre las 10 bandas Sentinel-2 y los 6 indices
core. Pares con `|corr| > 0.85` quedan flaggeados para colapso en EPIC 3.

In [ ]:
if not df_idx.is_empty() and indices_present:
    corr_pearson = correlation_pair(df_idx, cols_a=PASTIS_S2_BANDS, cols_b=indices_present, method="pearson")
    corr_spearman = correlation_pair(df_idx, cols_a=PASTIS_S2_BANDS, cols_b=indices_present, method="spearman")
    print("Top pares por |corr| (Pearson):")
    print(corr_pearson.head(8))
    redundant = corr_pearson.filter(pl.col("abs_corr") > 0.85)
    print(f"Pares redundantes (|r|>0.85): {redundant.height}")
else:
    corr_pearson = pl.DataFrame()
    corr_spearman = pl.DataFrame()
    redundant = pl.DataFrame()

In [ ]:
# Heatmaps con matplotlib directo (correlation_heatmap esta diseñado para matriz cuadrada).
def _plot_pair_heatmap(df_corr: pl.DataFrame, title: str, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 6), dpi=200)
    if df_corr.is_empty():
        ax.text(0.5, 0.5, "Sin datos", ha="center", va="center")
    else:
        rows = sorted(set(df_corr["feature_a"].to_list()))
        cols = sorted(set(df_corr["feature_b"].to_list()))
        ri = {r: i for i, r in enumerate(rows)}
        ci = {c: j for j, c in enumerate(cols)}
        mat = np.full((len(rows), len(cols)), np.nan, dtype=np.float64)
        for r in df_corr.iter_rows(named=True):
            mat[ri[r["feature_a"]], ci[r["feature_b"]]] = r["corr"]
        im = ax.imshow(mat, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
        ax.set_xticks(range(len(cols)))
        ax.set_xticklabels(cols, rotation=45, ha="right")
        ax.set_yticks(range(len(rows)))
        ax.set_yticklabels(rows)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                v = mat[i, j]
                if np.isfinite(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="black")
        fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(title)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

_plot_pair_heatmap(corr_pearson, "Correlacion Pearson: bandas vs indices", FIGURES / "sec3_corr_pearson_bands_indices.png")
_plot_pair_heatmap(corr_spearman, "Correlacion Spearman: bandas vs indices", FIGURES / "sec3_corr_spearman_bands_indices.png")

## Seccion 4 - VIF multicolinealidad (AC-3)

VIF sobre el conjunto bandas + indices con pre-filtro de redundancia
perfecta (|corr| > 0.99). Threshold convencional: VIF > 5 = warning, > 10 = drop.

In [ ]:
vif_cols = [c for c in PASTIS_S2_BANDS if c in df_idx.columns] + indices_present
if not df_idx.is_empty() and vif_cols:
    vif_df = vif_table(df_idx, cols=vif_cols)
    print(vif_df)
    to_drop = vif_df.filter(pl.col("status").is_in(["drop", "dropped_near_perfect_corr"]))
    print(f"Features marcados como drop: {to_drop.height}")
else:
    vif_df = pl.DataFrame()
vif_barplot(vif_df, out_path=FIGURES / "sec4_vif_barplot.png")

## Seccion 5 - Pairplot top-5 features por clase (AC-4)

Subset top-5 features para discriminar cultivos (heuristica fija): NDVI,
NDWI, NDMI, NDRE, EVI. Cap clases a top-10 + subsample 2000 por clase.

In [ ]:
top5_features = [f for f in ["NDVI", "NDWI", "NDMI", "NDRE", "EVI"] if f in df_idx.columns]
pairplot_by_class(
    df=df_idx,
    features=top5_features,
    class_col="class_name",
    out_path=FIGURES / "sec5_pairplot_top5_by_class.png",
    subsample_per_class=2000,
    top_classes=10,
    seed=seed,
)
print(f"Pairplot generado con features: {top5_features}")

## Seccion 6 - Pico NDVI por clase + timing (AC-5)

Pico NDVI detectado por parcela-pixel. Boxplot del valor pico y violin del
mes pico segmentado por clase.

In [ ]:
if not df_idx.is_empty() and "NDVI" in df_idx.columns:
    df_ts = df_idx.rename({"NDVI": "ndvi", "date_parsed": "date_d"}).filter(
        pl.col("class_id").is_between(1, 18)
    )
    df_peaks = phenology_peaks(df_ts, parcel_col="parcel_id", date_col="date_d", ndvi_col="ndvi", class_col="class_name")
    print(f"Peaks detectados: {df_peaks.height}")
    print(df_peaks.head(8))
else:
    df_peaks = pl.DataFrame()

In [ ]:
# Box NDVI peak por clase + violin mes pico
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=200)
if not df_peaks.is_empty():
    pdf = df_peaks.to_pandas()
    classes_order = (
        df_peaks.group_by("class_name").agg(pl.len().alias("n")).sort("n", descending=True)["class_name"].to_list()
    )
    # Boxplot horizontal de peak_ndvi_value
    data_box = [pdf.loc[pdf["class_name"] == c, "peak_ndvi_value"].dropna().to_numpy() for c in classes_order]
    axes[0].boxplot(data_box, vert=False, labels=classes_order, showfliers=False)
    axes[0].set_title("Pico NDVI por clase")
    axes[0].set_xlabel("NDVI pico")
    # Violin de mes pico
    data_violin = [pdf.loc[pdf["class_name"] == c, "peak_ndvi_month"].dropna().to_numpy() for c in classes_order]
    valid = [d for d in data_violin if d.size > 0]
    if valid:
        axes[1].violinplot(valid, vert=False, showmeans=True)
        axes[1].set_yticks(np.arange(1, len(classes_order) + 1))
        axes[1].set_yticklabels(classes_order)
        axes[1].set_title("Mes del pico NDVI por clase")
        axes[1].set_xlabel("Mes (1-12)")
        axes[1].set_xlim(0.5, 12.5)
else:
    for ax in axes:
        ax.text(0.5, 0.5, "Sin datos PASTIS-R", ha="center", va="center")
fig.tight_layout()
fig.savefig(FIGURES / "sec6_peak_ndvi_by_class.png", dpi=200, bbox_inches="tight")
plt.close(fig)

## Seccion 7 - ACF/PACF temporal por parcela (AC-6)

Resampleo mensual + interpolacion linear ANTES de ACF/PACF (dates-S2 es
irregular, median ~5 dias). `max_lag=6` por cobertura PASTIS ~14 meses.

In [ ]:
if not df_idx.is_empty() and "NDVI" in df_idx.columns:
    df_acf_input = df_idx.rename({"NDVI": "ndvi", "date_parsed": "date_d"})
    df_acf = acf_pacf_per_parcel(
        df_acf_input,
        max_lag=max_lag,
        parcel_col="parcel_id",
        date_col="date_d",
        series_col="ndvi",
        class_col="class_name",
    )
    print(f"Filas ACF/PACF: {df_acf.height}")
    print(df_acf.head(8))
else:
    df_acf = pl.DataFrame()
acf_grid_by_class(df_acf, out_path=FIGURES / "sec7_acf_grid_by_class.png", ncols=3)

## Seccion 8 - DTW clustering temporal (AC-7)

DTW con `tslearn.TimeSeriesKMeans` y Sakoe-Chiba band (radius=3) para limitar
coste O(T*radius). Z-normalizacion por parcela. Tabla cruzada `cluster x clase`
para identificar mono-cultivo vs doble ciclo.

In [ ]:
if not df_idx.is_empty() and "NDVI" in df_idx.columns:
    df_dtw_input = df_idx.rename({"NDVI": "ndvi", "date_parsed": "date_d"})
    df_clusters, km_model = dtw_cluster_temporal(
        df_dtw_input,
        n_clusters=n_dtw_clusters,
        parcel_col="parcel_id",
        date_col="date_d",
        series_col="ndvi",
        class_col="class_name",
        sakoe_chiba_radius=3,
        seed=seed,
    )
    print(f"Clusters asignados a {df_clusters.height} parcelas")
    if not df_clusters.is_empty():
        cross = (
            df_clusters.group_by(["cluster_id", "class_name"])
            .agg(pl.len().alias("n"))
            .sort(["cluster_id", "n"], descending=[False, True])
        )
        print("Tabla cluster x clase:")
        print(cross.head(20))
else:
    df_clusters = pl.DataFrame()
    km_model = None
dtw_centroids_plot(km_model, df_clusters, out_path=FIGURES / "sec8_dtw_centroids.png")

## Seccion 9 - ERA5 anomalias 2018-2020 cruzado con NDVI (AC-8 opcional)

Requiere auth GEE (`earthengine authenticate`). Si falla, devuelve df vacio
y skipea el plot con mensaje claro. La ROI usada es el bbox global de los
patches PASTIS-R cargados.

In [ ]:
ee_ok = False
df_era5 = pl.DataFrame(schema={"year": pl.Int64, "roi_name": pl.Utf8, "precip_mm": pl.Float64})
df_ndvi_annual = pl.DataFrame(schema={"year": pl.Int64, "roi_name": pl.Utf8, "ndvi_max": pl.Float64})
if not patch_coords.is_empty():
    try:
        import ee  # type: ignore[import-untyped]
        from ml.ingest.gee_sampler import init_ee
        init_ee()
        ee_ok = True
    except Exception as exc:  # noqa: BLE001
        print(f"AC-8 ERA5 skipped: GEE no disponible ({exc})")
if ee_ok and not patch_coords.is_empty():
    lons = patch_coords["lon"].to_numpy()
    lats = patch_coords["lat"].to_numpy()
    bbox = [float(np.min(lons)), float(np.min(lats)), float(np.max(lons)), float(np.max(lats))]
    try:
        roi = ee.Geometry.Rectangle(bbox)  # type: ignore[name-defined]
        df_era5 = era5_annual_precip(roi=roi, years=era5_years, cache_path=CACHE, roi_name="pastis_bbox")
    except Exception as exc:  # noqa: BLE001
        print(f"era5_annual_precip fallo: {exc}")
print(f"ERA5 rows: {df_era5.height}")
if not df_idx.is_empty() and "NDVI" in df_idx.columns:
    df_ndvi_annual = (
        df_idx.with_columns(pl.col("date_parsed").dt.year().alias("year"))
        .group_by("year")
        .agg(pl.col("NDVI").max().alias("ndvi_max"))
        .with_columns(pl.lit("pastis_bbox").alias("roi_name"))
        .select(["year", "roi_name", "ndvi_max"])
    )
print(f"NDVI annual rows: {df_ndvi_annual.height}")

In [ ]:
df_anomaly = era5_ndvi_anomaly(df_era5, df_ndvi_annual)
print(df_anomaly)
dual_axis_precip_ndvi(df_era5, df_ndvi_annual, out_path=FIGURES / "sec9_era5_ndvi_dual_axis.png")

## Seccion 10 - Conclusiones para EPIC 3 (AC-10)

Mapeo decisiones para feature engineering downstream (US-014) y baseline
XGBoost (US-017):

1. **Features redundantes a colapsar**: revisar `corr_pearson.filter(|corr|>0.85)`
   y `vif_df.status in {"drop", "dropped_near_perfect_corr"}`. NDVI y EVI
   suelen quedar fuertemente correlacionados; en el subset core se conserva
   NDVI y se descarta EVI (o se reemplaza por SAVI cuando hay suelo expuesto).
2. **Timing pico NDVI por clase**: `df_peaks.group_by("class_name").agg(median peak_ndvi_month)`
   da el calendario fenologico. En US-014 se deriva la feature `peak_month_distance_to_class_median`.
3. **Features fenologicas a derivar**: usar `df_clusters` para etiquetar cada
   parcela como `mono` o `dual` (cluster con 1 vs 2 picos en el centroide).
   La etiqueta entra como feature categorica en el baseline.
4. **Normalizacion temporal**: el resampleo mensual + interpolacion linear
   usado para ACF/PACF debe replicarse en US-014 antes de cualquier feature
   temporal (FFT, slopes). Z-score por parcela quita el bias de magnitud.
5. **AC-8 ERA5**: si `df_anomaly.is_empty()`, dejar marcado como limitacion
   en el handoff y considerar como opcional-bonus.

In [ ]:
# Resumen final
summary = {
    "patches_loaded": len(patch_ids_sample),
    "parcelas_pixel": df_idx.select("parcel_id").n_unique() if not df_idx.is_empty() else 0,
    "indices_computed": indices_present,
    "pares_redundantes_pearson_gt_085": redundant.height if not redundant.is_empty() else 0,
    "features_vif_drop": (vif_df.filter(pl.col("status").is_in(["drop", "dropped_near_perfect_corr"])).height if not vif_df.is_empty() else 0),
    "peaks_detected": df_peaks.height if not df_peaks.is_empty() else 0,
    "dtw_clusters_assigned": df_clusters.height if not df_clusters.is_empty() else 0,
    "era5_years": df_era5.height if not df_era5.is_empty() else 0,
    "figures_dir": str(FIGURES),
}
for k, v in summary.items():
    print(f"{k}: {v}")